In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files
import time
from abc import ABC, abstractmethod
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              AdaBoostClassifier, GradientBoostingClassifier)
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import seaborn as sns
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


class CancerClassifier(ABC):

    def __init__(self, csv_filepath=None):
        self.csv_file      = csv_filepath
        self.df            = None
        self.custom_models = 0
        self.trained_models = []
        self.results        = {}
        self.predictions    = {}
        self.train_times    = {}
        self.models = {
            "Logistic Regression": LogisticRegression(
                                       max_iter=1000,
                                       class_weight='balanced'),
            "Decision Tree":       DecisionTreeClassifier(
                                       random_state=42,
                                       max_depth=10,
                                       min_samples_leaf=5,
                                       class_weight='balanced'),
            "Random Forest":       RandomForestClassifier(
                                       n_estimators=200,
                                       random_state=42,
                                       max_depth=10,
                                       min_samples_leaf=5,
                                       class_weight='balanced'),
            "Extra Trees":         ExtraTreesClassifier(
                                       n_estimators=200,
                                       random_state=42,
                                       max_depth=10,
                                       min_samples_leaf=5,
                                       class_weight='balanced'),
            "AdaBoost":            AdaBoostClassifier(
                                       n_estimators=150,
                                       random_state=42),
            "Gradient Boosting":   GradientBoostingClassifier(
                                       n_estimators=150,
                                       random_state=42,
                                       max_depth=4,
                                       learning_rate=0.05,
                                       subsample=0.8),
            "SVM (RBF Kernel)":    SVC(
                                       kernel='rbf',
                                       probability=True,
                                       random_state=42,
                                       class_weight='balanced'),
            "XGBoost":             XGBClassifier(
                                       n_estimators=150,
                                       random_state=42,
                                       max_depth=4,
                                       learning_rate=0.05,
                                       subsample=0.8,
                                       reg_alpha=0.1,
                                       reg_lambda=1.0,
                                       eval_metric='logloss'),
            "LightGBM":            LGBMClassifier(
                                       n_estimators=150,
                                       random_state=42,
                                       max_depth=6,
                                       learning_rate=0.05,
                                       reg_alpha=0.1,
                                       reg_lambda=1.0,
                                       min_child_samples=20,
                                       class_weight='balanced'),
            "CatBoost":            CatBoostClassifier(
                                       verbose=0,
                                       random_state=42,
                                       depth=6,
                                       learning_rate=0.05,
                                       l2_leaf_reg=3.0)
        }
        self.default_models = {
            "Logistic Regression": lambda: LogisticRegression(
                                       max_iter=1000,
                                       class_weight='balanced'),
            "Decision Tree":       lambda: DecisionTreeClassifier(
                                       random_state=42,
                                       max_depth=10,
                                       min_samples_leaf=5,
                                       class_weight='balanced'),
            "Random Forest":       lambda: RandomForestClassifier(
                                       n_estimators=200,
                                       random_state=42,
                                       max_depth=10,
                                       min_samples_leaf=5,
                                       class_weight='balanced'),
            "Extra Trees":         lambda: ExtraTreesClassifier(
                                       n_estimators=200,
                                       random_state=42,
                                       max_depth=10,
                                       min_samples_leaf=5,
                                       class_weight='balanced'),
            "AdaBoost":            lambda: AdaBoostClassifier(
                                       n_estimators=150,
                                       random_state=42),
            "Gradient Boosting":   lambda: GradientBoostingClassifier(
                                       n_estimators=150,
                                       random_state=42,
                                       max_depth=4,
                                       learning_rate=0.05,
                                       subsample=0.8),
            "SVM (RBF Kernel)":    lambda: SVC(
                                       kernel='rbf',
                                       probability=True,
                                       random_state=42,
                                       class_weight='balanced'),
            "XGBoost":             lambda: XGBClassifier(
                                       n_estimators=150,
                                       random_state=42,
                                       max_depth=4,
                                       learning_rate=0.05,
                                       subsample=0.8,
                                       reg_alpha=0.1,
                                       reg_lambda=1.0,
                                       eval_metric='logloss'),
            "LightGBM":            lambda: LGBMClassifier(
                                       n_estimators=150,
                                       random_state=42,
                                       max_depth=6,
                                       learning_rate=0.05,
                                       reg_alpha=0.1,
                                       reg_lambda=1.0,
                                       min_child_samples=20,
                                       class_weight='balanced'),
            "CatBoost":            lambda: CatBoostClassifier(
                                       verbose=0,
                                       random_state=42,
                                       depth=6,
                                       learning_rate=0.05,
                                       l2_leaf_reg=3.0)
        }

    def upload_filepath(self, csv_filepath):
        self.csv_file = csv_filepath
        self.df       = pd.read_csv(self.csv_file)
        print(f"{self.csv_file}: Loaded {self.df.shape}")

    def train_test_split(self, target_col="Risk_Level", test_size=0.2, random_state=42):
        if self.df is None:
            raise ValueError(f"{self.name}: No data loaded. "
                             f"Run upload_filepath() first.")
        if target_col not in self.df.columns:
            raise ValueError(f"{self.name}: Target column '{target_col}' "
                             f"not found. Available: "
                             f"{self.df.columns.tolist()}")

        self.target_col = target_col
        self.le         = LabelEncoder().fit(self.df[target_col])
        X               = self.df[self.core_features]
        y               = self.le.transform(self.df[target_col])

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state
        )
        scaler       = StandardScaler()
        self.X_train = scaler.fit_transform(self.X_train)
        self.X_test  = scaler.transform(self.X_test)

        print(f"{self.name}: Split complete")
        print(f"  Target:    {target_col}")
        print(f"  Train:     {self.X_train.shape[0]} rows")
        print(f"  Test:      {self.X_test.shape[0]} rows")
        print(f"  Test size: {test_size*100:.0f}%")


    @abstractmethod
    def train(self, model_name="XGBoost"):
        pass

    def pass_model(self, model, model_name=None):
        if not hasattr(model, 'fit') or not hasattr(model, 'predict'):
            raise TypeError("Model must implement fit() and predict() methods.")
        if model_name is None:
            model_name = type(model).__name__
        if model_name in self.models:
            base_name = model_name
            counter   = 1
            while model_name in self.models:
                model_name = f"{base_name}_{counter}"
                counter   += 1
            print(f"Name already taken — model saved as '{model_name}'.")
        self.models[model_name] = model
        self.custom_models += 1
        print(f"Model '{model_name}' added successfully.")

    # ── Param setters ──────────────────────────────────────────────────────
    def set_logistic_regression_params(self, **kwargs):
        current = self.models["Logistic Regression"].get_params()
        current.update(kwargs)
        self.models["Logistic Regression"] = LogisticRegression(**current)

    def set_decision_tree_params(self, **kwargs):
        current = self.models["Decision Tree"].get_params()
        current.update(kwargs)
        self.models["Decision Tree"] = DecisionTreeClassifier(**current)

    def set_random_forest_params(self, **kwargs):
        current = self.models["Random Forest"].get_params()
        current.update(kwargs)
        self.models["Random Forest"] = RandomForestClassifier(**current)

    def set_extra_trees_params(self, **kwargs):
        current = self.models["Extra Trees"].get_params()
        current.update(kwargs)
        self.models["Extra Trees"] = ExtraTreesClassifier(**current)

    def set_adaboost_params(self, **kwargs):
        current = self.models["AdaBoost"].get_params()
        current.update(kwargs)
        self.models["AdaBoost"] = AdaBoostClassifier(**current)

    def set_gradient_boosting_params(self, **kwargs):
        current = self.models["Gradient Boosting"].get_params()
        current.update(kwargs)
        self.models["Gradient Boosting"] = GradientBoostingClassifier(**current)

    def set_svm_params(self, **kwargs):
        current = self.models["SVM (RBF Kernel)"].get_params()
        current.update(kwargs)
        self.models["SVM (RBF Kernel)"] = SVC(**current)

    def set_xgboost_params(self, **kwargs):
        current = self.models["XGBoost"].get_params()
        current.update(kwargs)
        self.models["XGBoost"] = XGBClassifier(**current)

    def set_lightgbm_params(self, **kwargs):
        current = self.models["LightGBM"].get_params()
        current.update(kwargs)
        self.models["LightGBM"] = LGBMClassifier(**current)

    def set_catboost_params(self, **kwargs):
        current = self.models["CatBoost"].get_params()
        current.update(kwargs)
        self.models["CatBoost"] = CatBoostClassifier(**current)

    # ── Individual resets ──────────────────────────────────────────────────
    def reset_logistic_regression_params(self):
        self.models["Logistic Regression"] = \
            self.default_models["Logistic Regression"]()
        print("Logistic Regression reset to defaults.")

    def reset_decision_tree_params(self):
        self.models["Decision Tree"] = self.default_models["Decision Tree"]()
        print("Decision Tree reset to defaults.")

    def reset_random_forest_params(self):
        self.models["Random Forest"] = self.default_models["Random Forest"]()
        print("Random Forest reset to defaults.")

    def reset_extra_trees_params(self):
        self.models["Extra Trees"] = self.default_models["Extra Trees"]()
        print("Extra Trees reset to defaults.")

    def reset_adaboost_params(self):
        self.models["AdaBoost"] = self.default_models["AdaBoost"]()
        print("AdaBoost reset to defaults.")

    def reset_gradient_boosting_params(self):
        self.models["Gradient Boosting"] = \
            self.default_models["Gradient Boosting"]()
        print("Gradient Boosting reset to defaults.")

    def reset_svm_params(self):
        self.models["SVM (RBF Kernel)"] = \
            self.default_models["SVM (RBF Kernel)"]()
        print("SVM reset to defaults.")

    def reset_xgboost_params(self):
        self.models["XGBoost"] = self.default_models["XGBoost"]()
        print("XGBoost reset to defaults.")

    def reset_lightgbm_params(self):
        self.models["LightGBM"] = self.default_models["LightGBM"]()
        print("LightGBM reset to defaults.")

    def reset_catboost_params(self):
        self.models["CatBoost"] = self.default_models["CatBoost"]()
        print("CatBoost reset to defaults.")

    def reset_all_params(self):
        for name, factory in self.default_models.items():
            self.models[name] = factory()
        print("All models reset to defaults.")

    def test(self, model_name=None):
        # ── Select model ──────────────────────────────────
        if model_name is None:
            if not self.trained_models:
                raise ValueError(f"{self.name}: No models trained yet. "
                                 f"Run train() first.")
            model_name = self.trained_models[-1]
            print(f"No model specified — defaulting to last trained: "
                  f"{model_name}")

        if model_name not in self.trained_models:
            raise ValueError(f"'{model_name}' has not been trained yet. "
                             f"Trained models: {self.trained_models}")

        preds = self.predictions[model_name]

        # ── Classification Report ──────────────────────────
        print(f"\n{'='*60}")
        print(f"{self.name} — Test Results: {model_name}")
        print(f"{'='*60}")
        print(f"Accuracy:      {self.results[model_name]:.4f}")
        print(f"Training time: {self.train_times[model_name]:.2f}s")

        print(f"\n=== Classification Report ===")
        print(classification_report(self.y_test, preds,
                                    target_names=self.le.classes_))

        # ── Overfitting Check ──────────────────────────────
        model     = self.models[model_name]
        train_acc = accuracy_score(self.y_train,
                                   model.predict(self.X_train))
        test_acc  = self.results[model_name]
        gap       = train_acc - test_acc
        status    = ("⚠️ Overfit" if gap > 0.15 else
                     "⚡ Slight"  if gap > 0.05 else "✓ Good")

        print(f"\n=== Overfitting Check ===")
        print(f"Train Accuracy: {train_acc:.4f}")
        print(f"Test Accuracy:  {test_acc:.4f}")
        print(f"Gap:            {gap:.4f} — {status}")

        # ── Confusion Matrix ───────────────────────────────
        cm = confusion_matrix(self.y_test, preds)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=self.le.classes_,
                    yticklabels=self.le.classes_)
        plt.title(f"Confusion Matrix — {model_name}\n{self.name}",
                  weight='bold')
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.tight_layout()
        plt.show()

        # ── Accuracy Bar Chart ─────────────────────────────
        results_df = pd.DataFrame(
            list(self.results.items()),
            columns=["Model", "Accuracy"]
        ).sort_values(by="Accuracy", ascending=False)

        plt.figure(figsize=(10, 6))
        sns.barplot(x='Accuracy', y='Model', hue='Model',
                    data=results_df, palette="coolwarm", legend=False)
        plt.title(f"Model Accuracy — {self.name}", weight='bold')
        plt.xlabel("Accuracy")
        plt.ylabel("Model")
        plt.axvline(x=0.8, color='red', linestyle='--',
                    alpha=0.5, label='80% threshold')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.show()

In [ ]:
class ColonCancerClassifier(CancerClassifier):

    name = "Colon Cancer Classifier"

    def __init__(self, df=None):
        super().__init__(df=df)
        self.core_features = [
            "Age", "BMI", "Obesity", "Family_History",
            "Diet_Red_Meat", "Fruit_Veg_Intake",
            "Physical_Activity", "Calcium_Intake"
        ]

    def train(self, model_name="XGBoost", verbose=True):
        # ── Validate prerequisites ─────────────────────────
        if self.X_train is None:
            raise ValueError(f"{self.name}: No train/test split found. "
                             f"Run split() first.")

        if model_name not in self.models:
            raise ValueError(f"{self.name}: Model '{model_name}' not found. "
                             f"Available: {list(self.models.keys())}")

        # ── Validate core features exist in dataframe ──────
        missing = [f for f in self.core_features
                   if f not in self.df.columns]
        if missing:
            raise ValueError(f"{self.name}: Missing required features: "
                             f"{missing}")

        # ── Train ──────────────────────────────────────────
        model = self.models[model_name]
        start = time.time()

        if model_name == "XGBoost":
            sw = compute_sample_weight(class_weight='balanced',
                                       y=self.y_train)
            model.fit(self.X_train, self.y_train, sample_weight=sw)
        else:
            model.fit(self.X_train, self.y_train)

        elapsed = time.time() - start

        # ── Store results ──────────────────────────────────
        y_pred    = model.predict(self.X_test)
        test_acc  = accuracy_score(self.y_test, y_pred)
        train_acc = accuracy_score(self.y_train,
                                   model.predict(self.X_train))
        gap       = train_acc - test_acc
        status    = ("⚠️ Overfit" if gap > 0.15 else
                     "⚡ Slight"  if gap > 0.05 else "✓ Good")

        self.results[model_name]     = test_acc
        self.predictions[model_name] = y_pred
        self.train_times[model_name] = elapsed

        if model_name not in self.trained_models:
            self.trained_models.append(model_name)

        # ── Report ─────────────────────────────────────────
        if verbose:
            print(f"\n{self.name}: '{model_name}' trained in {elapsed:.2f}s")
            print(f"  Test Accuracy:  {test_acc:.4f}")
            print(f"  Train Accuracy: {train_acc:.4f}")
            print(f"  Gap:            {gap:.4f} — {status}")
            if gap > 0.15:
                print(f"  ⚠️  Warning: significant overfitting detected. "
                      f"Consider regularization or more data.")

In [ ]:
class SkinCancerClassifier(CancerClassifier):

    name = "Skin Cancer Classifier"

    def __init__(self, df=None):
        super().__init__()
        self.core_features = [
      'Age', 'Gender', 'Smoking', 'Alcohol_Use', 'Obesity',
      'Family_History', 'Diet_Red_Meat', 'Diet_Salted_Processed',
      'Fruit_Veg_Intake', 'Physical_Activity', 'Air_Pollution',
      'Occupational_Hazards', 'BRCA_Mutation',
      'Calcium_Intake', 'BMI', 'Physical_Activity_Level',

        ]

    def train(self, model_name="XGBoost", verbose=True):
        # ── Validate prerequisites ─────────────────────────
        if self.X_train is None:
            raise ValueError(f"{self.name}: No train/test split found. "
                             f"Run split() first.")

        if model_name not in self.models:
            raise ValueError(f"{self.name}: Model '{model_name}' not found. "
                             f"Available: {list(self.models.keys())}")

        # ── Validate core features exist in dataframe ──────
        missing = [f for f in self.core_features
                   if f not in self.df.columns]
        if missing:
            raise ValueError(f"{self.name}: Missing required features: "
                             f"{missing}")

        # ── Train ──────────────────────────────────────────
        model = self.models[model_name]
        start = time.time()

        if model_name == "XGBoost":
            sw = compute_sample_weight(class_weight='balanced',
                                       y=self.y_train)
            model.fit(self.X_train, self.y_train, sample_weight=sw)
        else:
            model.fit(self.X_train, self.y_train)

        elapsed = time.time() - start

        # ── Store results ──────────────────────────────────
        y_pred    = model.predict(self.X_test)
        test_acc  = accuracy_score(self.y_test, y_pred)
        train_acc = accuracy_score(self.y_train,
                                   model.predict(self.X_train))
        gap       = train_acc - test_acc
        status    = ("⚠️ Overfit" if gap > 0.15 else
                     "⚡ Slight"  if gap > 0.05 else "✓ Good")

        self.results[model_name]     = test_acc
        self.predictions[model_name] = y_pred
        self.train_times[model_name] = elapsed

        if model_name not in self.trained_models:
            self.trained_models.append(model_name)

        # ── Report ─────────────────────────────────────────
        if verbose:
            print(f"\n{self.name}: '{model_name}' trained in {elapsed:.2f}s")
            print(f"  Test Accuracy:  {test_acc:.4f}")
            print(f"  Train Accuracy: {train_acc:.4f}")
            print(f"  Gap:            {gap:.4f} — {status}")
            if gap > 0.15:
                print(f"  ⚠️  Warning: significant overfitting detected. "
                      f"Consider regularization or more data.")

